In [9]:
import ollama
import pickle
import numpy as np

In [10]:
filepath = r"C:\Users\Client\Desktop\LUMINAR TECHNOLAB\ollama\ml.txt"
with open(filepath,"r",encoding="utf-8") as f:
    text = f.read()
# chunk_size = 2000
# chunks = [text[i:i+chunk_size] for i in range(0,len(text),chunk_size)]
# chunks

def chunk_text(text, chunk_size=150):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunks.append(" ".join(words[i:i+chunk_size]))
    return chunks

chunks = chunk_text(text)


In [11]:
len(chunks)

7

In [12]:
embeddings = []
for chunk in chunks:
    response = ollama.embeddings(model='nomic-embed-text',prompt=chunk)
    embeddings.append(response["embedding"])
embeddings

[[0.10730178654193878,
  1.7918620109558105,
  -3.7716851234436035,
  0.1821216642856598,
  2.1196320056915283,
  -0.05082757771015167,
  1.0751597881317139,
  -0.6043163537979126,
  -0.2106570601463318,
  0.1138162612915039,
  -0.5514752268791199,
  1.2624735832214355,
  2.302260637283325,
  0.19598771631717682,
  -0.8620234727859497,
  -0.9849556684494019,
  -0.5285614132881165,
  -1.2335156202316284,
  -0.7696429491043091,
  0.2106650173664093,
  0.5419557094573975,
  0.18309110403060913,
  0.0806487426161766,
  -0.2993127405643463,
  1.4480501413345337,
  -0.1233559101819992,
  -0.5121679902076721,
  -0.374205082654953,
  0.4521685838699341,
  0.23592926561832428,
  0.2538891136646271,
  -0.7270335555076599,
  0.17083224654197693,
  0.025120049715042114,
  -1.9403069019317627,
  -0.08388499915599823,
  1.6029943227767944,
  0.09628026932477951,
  0.7170938849449158,
  0.6189330220222473,
  -0.14537313580513,
  0.8861886858940125,
  0.22925937175750732,
  -0.5165797472000122,
  0.63

In [13]:
data_to_save = {
    "chunks":chunks,
    "embeddings":embeddings
}

In [14]:
output_pickle = "ml.pkl"
with open(output_pickle,"wb") as f:
    pickle.dump(data_to_save,f)

In [15]:
def cosine_similarity(v1,v2):
    return np.dot(v1,v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

In [16]:
k = 8
pickle_file = r"C:\Users\Client\Desktop\LUMINAR TECHNOLAB\ollama\ml.pkl"
with open(pickle_file,'rb') as f:
    data = pickle.load(f)

chunks = data['chunks']
doc_embeddings = data["embeddings"]

while True:
    user_query = input("Enter the query (or type 'exit' to quit): ")
    
    if user_query.lower() == 'exit':
        print("Goodbye!")
        break

    response = ollama.embeddings(
        model = "nomic-embed-text",
        prompt = user_query
    )

    query_embedding = response["embedding"]

    similarities = [cosine_similarity(query_embedding,emb) for emb in doc_embeddings]

    top_k_indices = np.argsort(similarities)[-k:][::-1]

    retrieved_chunks = [chunks[i] for i in top_k_indices]

    # print("\n--- Retrieved Chunks ---")
    # for chunk in retrieved_chunks:
    #     print(chunk[:200])

    if not retrieved_chunks:
        print("No relevant information found.")
        exit()

    combined_context = "\n---\n".join(retrieved_chunks)

    prompt = f"""Answer the question using ONLY the context below.

    Give a COMPLETE answer.
    If multiple points exist (like types, steps, etc.), include ALL of them.

    If not present, say: I don't have information about that.

    CONTEXT:
    {combined_context}

    QUESTION: {user_query}
    """

    output = ollama.generate(
        model="gemma3:1b",
        prompt=prompt,
        options={"temperature": 0.0}  # Zero temperature = most strict
    )
    print(output['response'])

Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It involves building mathematical models based on training data to make predictions or decisions.

Here’s the answer to your question, based solely on the provided text:

**Types of Machine Learning:**

*   **Supervised Learning:** The algorithm learns from labeled training data.
*   **Unsupervised Learning:** The algorithm works on unlabeled data and finds hidden patterns.
*   **Reinforcement Learning:** The algorithm learns by interacting with an environment, receiving rewards or penalties.


TensorFlow is a deep learning framework developed by Google. It’s used for building and training machine learning models, particularly neural networks. It provides tools and libraries for data manipulation, model building, and deployment.

I don't have information about what an operating system is.
Goodbye!
